# 04-01. Why Four-box?  
# なぜ 4-box モデルが必要なのか

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

---

## 今日の目的 / Goals

この Notebook では、3-box モデルから 4-box モデルへ拡張する理由を、手を動かしながら理解します。  
In this notebook, we learn why we extend the 3-box model to a 4-box model, by running small experiments.

ここでの主役は、**北大西洋 N box** です。  
The main new character is the **North Atlantic N box**.

3-box では、高緯度表層を 1 つの箱で表していました。  
In the 3-box model, the high-latitude surface ocean was represented by one box.

4-box では、それを

```text
Southern Ocean / 南大洋 H
North Atlantic / 北大西洋 N
```

に分けます。

This split lets us ask a new scientific question:

> 深層水が作られる場所と、深層水が湧昇する場所を分けると、大気 CO2 はどう変わるのか？  
> What happens to atmospheric CO2 if the sinking region and the upwelling region are separated?

この Notebook の流れは次です。  
The structure is:

```text
3-box の限界
↓
世界地図イメージ
↓
4-box の概念
↓
簡単な Python モデル
↓
感度実験
↓
考察
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

plt.rcParams["figure.figsize"] = (7, 4)

## 1. 3-box モデルの限界 / Limitation of the 3-box model

3-box モデルでは、海洋を次の 3 つに分けました。  
In the 3-box model, the ocean was divided into three boxes.

```text
H: high-latitude surface / 高緯度表層
L: low-latitude surface  / 低緯度表層
D: deep ocean            / 深層
```

概念図はこうでした。  
The conceptual diagram was:

```text
Atmosphere
  ↑↓              ↑↓
High latitude H   Low latitude L
       \          /
          Deep D
```

このモデルで表現できることは増えました。  
This model can represent more than the 2-box model.

しかし、まだ大きな問題があります。  
But it still has a major problem.

**高緯度が 1 つしかない**ことです。  
There is only **one high-latitude box**.

現実の海洋では、高緯度には少なくとも重要な 2 つの役割があります。  
In the real ocean, high latitudes have at least two important roles.

```text
North Atlantic:
  deep-water formation / 深層水形成

Southern Ocean:
  deep-water upwelling and gas exchange / 深層水の湧昇とガス交換
```

3-box では、この 2 つを区別できません。  
The 3-box model cannot distinguish these two roles.

## 2. 世界地図を頭に置く / Keep a world map in mind

非常に粗い世界地図を描くと、次のように考えられます。  
A very rough world-map view is:

```text
          North Atlantic
                N
                ↓ sinking

Low-latitude ocean L  → warm, productive

          Southern Ocean
                H
                ↑ upwelling

          Deep Ocean D
```

4-box モデルでは、3-box の H を分割します。  
In the 4-box model, the old H box of the 3-box model is split.

```text
3-box:
  H = generic high latitude

4-box:
  H = Southern Ocean / 南大洋
  N = North Atlantic / 北大西洋
```

ここで重要なのは、箱の数が増えたことではありません。  
The important point is not simply that the number of boxes increased.

重要なのは、**別々の物理過程を別々の箱に割り当てられるようになったこと**です。  
The important point is that **different physical processes can now be assigned to different boxes**.

In [ ]:
# A simple schematic map using matplotlib.
# This is not a real geographic map. It is a teaching diagram.

fig, ax = plt.subplots(figsize=(7, 5))

boxes = {
    "N\\nNorth Atlantic": (0.5, 0.82),
    "L\\nLow latitude": (0.5, 0.52),
    "H\\nSouthern Ocean": (0.5, 0.22),
    "D\\nDeep ocean": (0.5, 0.02),
}

for label, (x, y) in boxes.items():
    ax.text(x, y, label, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black"))

# arrows
ax.annotate("", xy=(0.5, 0.73), xytext=(0.5, 0.77), arrowprops=dict(arrowstyle="->"))
ax.text(0.58, 0.75, "sinking", va="center")

ax.annotate("", xy=(0.5, 0.31), xytext=(0.5, 0.11), arrowprops=dict(arrowstyle="->"))
ax.text(0.58, 0.21, "upwelling", va="center")

ax.annotate("", xy=(0.5, 0.43), xytext=(0.5, 0.31), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(0.5, 0.61), xytext=(0.5, 0.73), arrowprops=dict(arrowstyle="->"))

ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 0.95)
ax.axis("off")
ax.set_title("Schematic 4-box ocean / 4-box 海洋模式図")
plt.show()

### Mini exercise 1 / ミニ演習 1

次の問いを考えてください。  
Think about the following questions.

1. 深層水形成に近い箱はどれですか。  
   Which box is closest to deep-water formation?

2. 深層水の湧昇に近い箱はどれですか。  
   Which box is closest to deep-water upwelling?

3. 大気 CO2 に強く効きそうなのはどちらですか。  
   Which one may strongly affect atmospheric CO2?

**想定される考え方 / Expected thinking**

- N box は北大西洋で、沈み込みを表す。  
  N represents the North Atlantic and sinking.

- H box は南大洋で、湧昇と大気海洋 CO2 交換を表す。  
  H represents the Southern Ocean and upwelling plus air-sea CO2 exchange.

## 3. 4-box の箱と変数 / Boxes and variables in the 4-box model

この Notebook では、4 つの箱を使います。  
In this notebook, we use four boxes.

| Box | 日本語 | English | 役割 |
|---|---|---|---|
| H | 南大洋表層 | Southern Ocean surface | 深層水の湧昇・ガス交換 |
| L | 低緯度表層 | Low-latitude surface | 暖かい表層・生物生産 |
| N | 北大西洋表層 | North Atlantic surface | 深層水形成 |
| D | 深層 | Deep ocean | 炭素・栄養塩の貯蔵 |

プログラムの変数名は、箱名を最後につけます。  
In the program, the box name is appended to the variable name.

```text
PO4H: phosphate in H
PO4L: phosphate in L
PO4N: phosphate in N
PO4D: phosphate in D
```

同じように、

```text
DICH, DICL, DICN, DICD
ALKH, ALKL, ALKN, ALKD
DO2H, DO2L, DO2N, DO2D
```

を使います。  
We use the same naming rule for DIC, ALK, and O2.

In [ ]:
# Conversion factors
CV1 = 1.0250e3    # mol/kg -> mol/m3
CV2 = 9.7561e-4   # mol/m3 -> mol/kg
CV3 = 1.0e6       # atm -> ppmv
CV4 = 3.1536e7    # seconds/year
CV5 = 8.64e4      # seconds/day

def to_umolkg(x):
    return CV2 * 1.0e6 * x

def to_ppmv(x):
    return CV3 * x

## 4. まずボックスの大きさを決める / Define box sizes

Toggweiler 型の 4-box モデルでは、北大西洋 N と南大洋 H の面積を別々にします。  
In a Toggweiler-type 4-box model, the North Atlantic N and Southern Ocean H have separate areas.

ここでは教育用に、次のような比率を使います。  
For teaching purposes, we use the following fractions.

```text
H: 10% of ocean area
N: 15% of ocean area
L: 75% of ocean area
```

H と N は厚さ 250 m、L は厚さ 100 m とします。  
H and N have 250 m thickness, while L has 100 m thickness.

In [ ]:
VOCN_TOTAL = 1.292e18  # total ocean volume [m3]
AOCN = 3.49e14         # ocean area [m2]
VATM = 1.773e20

FH_area = 0.10
FN_area = 0.15
FL_area = 1.0 - FH_area - FN_area

ZH = 250.0
ZN = 250.0
ZL = 100.0

AOCNH = AOCN * FH_area
AOCNN = AOCN * FN_area
AOCNL = AOCN * FL_area

VOCNH = AOCNH * ZH
VOCNN = AOCNN * ZN
VOCNL = AOCNL * ZL
VOCND = VOCN_TOTAL - VOCNH - VOCNN - VOCNL

pd.DataFrame({
    "Box": ["H Southern Ocean", "L Low latitude", "N North Atlantic", "D Deep"],
    "Area fraction": [FH_area, FL_area, FN_area, np.nan],
    "Volume [m3]": [VOCNH, VOCNL, VOCNN, VOCND],
    "Volume fraction": [VOCNH/VOCN_TOTAL, VOCNL/VOCN_TOTAL, VOCNN/VOCN_TOTAL, VOCND/VOCN_TOTAL],
})

### Mini exercise 2 / ミニ演習 2

N box の面積割合を 15% から 5% にしたら、N box の体積はどう変わるでしょうか。  
If the N-box area fraction is changed from 15% to 5%, how does the N-box volume change?

実行する前に予想してください。  
Predict before running.

In [ ]:
for fn in [0.05, 0.10, 0.15, 0.20]:
    a_n = AOCN * fn
    v_n = a_n * ZN
    print(f"FN_area = {fn:4.2f}, VOCNN = {v_n:.3e} m3, fraction = {v_n/VOCN_TOTAL:.4f}")

## 5. 温度・塩分・初期値 / Temperature, salinity, and initial values

4-box では、H, L, N, D の温度を分けます。  
In the 4-box model, H, L, N, and D have different temperatures.

ここでは簡単に次のようにします。  
Here we use simple values:

```text
H: 1.0 deg C   Southern Ocean
L: 21.5 deg C  Low latitude
N: 3.0 deg C   North Atlantic
D: 1.75 deg C  Deep ocean
```

同じ DIC・ALK でも、温度が違うと pCO2 が変わります。  
Even with the same DIC and ALK, pCO2 differs if temperature differs.

これは 4-box で地域を分ける意味の一つです。  
This is one reason why regional boxes matter.

In [ ]:
TEMP = {"H": 1.0, "L": 21.5, "N": 3.0, "D": 1.75}
SALT = {"H": 34.7, "L": 34.7, "N": 34.7, "D": 34.7}

def initial_four_box():
    x = {}
    for b in ["H", "L", "N", "D"]:
        x[f"PO4{b}"] = 2.10e-6 * CV1
        x[f"DIC{b}"] = 2.235e-3 * CV1
        x[f"ALK{b}"] = 2.374e-3 * CV1
        x[f"DO2{b}"] = 1.60e-4 * CV1
    x["PCO2A"] = 280.0 / CV3
    return x

x = initial_four_box()

pd.DataFrame({
    "Box": ["H", "L", "N", "D"],
    "Temperature [deg C]": [TEMP[b] for b in ["H", "L", "N", "D"]],
    "PO4 [umol/kg]": [to_umolkg(x[f"PO4{b}"]) for b in ["H", "L", "N", "D"]],
    "DIC [umol/kg]": [to_umolkg(x[f"DIC{b}"]) for b in ["H", "L", "N", "D"]],
    "ALK [ueq/kg]": [to_umolkg(x[f"ALK{b}"]) for b in ["H", "L", "N", "D"]],
    "O2 [umol/kg]": [to_umolkg(x[f"DO2{b}"]) for b in ["H", "L", "N", "D"]],
})

## 6. 炭酸系と酸素飽和 / Carbonate chemistry and oxygen saturation

ここでは 2-box / 3-box と同じ補助関数を使います。  
Here we use the same helper functions as in the 2-box and 3-box notebooks.

関数の中身は長いですが、考え方は単純です。  
The functions are long, but the idea is simple.

```text
DIC + ALK + Temperature + Salinity
↓
pCO2
```

```text
Temperature + Salinity
↓
O2 saturation
```

In [ ]:
def o2sat(TEM, SAL):
    N1 = -1.734292e2
    N2 = +2.496339e2
    N3 = +1.433483e2
    N4 = -2.184920e1
    N5 = -3.309600e-2
    N6 = +1.425900e-2
    N7 = -1.700000e-3
    ATEM = TEM + 273.15
    O2S = math.exp(
        N1 + N2 * 1.0e2 / ATEM
        + N3 * math.log(ATEM / 1.0e2)
        + N4 * ATEM / 1.0e2
        + SAL * (N5 + N6 * ATEM / 1.0e2 + N7 * (ATEM / 1.0e2) ** 2)
    )
    return O2S * 4.35e1 * 1.025e-3

def chemeq_const(TEM, SAL):
    ATEM = TEM + 273.15
    TK = ATEM * 1.0e-2
    SK = 2.3517e-2 + (-2.3656e-2 + 4.7036e-3 * TK) * TK
    K0 = math.exp(-6.02409e1 + 9.34517e1 / TK + 2.33585e1 * math.log(TK) + SK * SAL)
    K1 = math.exp(math.log(10.0) * (
        13.7201 - 3.1334e-2 * ATEM - 3.23576e3 / ATEM
        - 1.3e-5 * SAL * ATEM + 1.032e-1 * math.sqrt(SAL)
    ))
    safe_sal = SAL if SAL > 0 else 1.0
    K2 = math.exp(math.log(10.0) * (
        -5.3719645e3 - 1.671221e0 * ATEM - 2.2913e-1 * SAL
        - 1.83802e1 * math.log10(safe_sal) + 1.2837528e5 / ATEM
        + 2.1943005e3 * math.log10(ATEM) + 8.0944e-4 * SAL * ATEM
        + 5.61711e3 * math.log10(safe_sal) / ATEM - 2.136e0 * SAL / ATEM
    ))
    KB = math.exp(math.log(10.0) * (-9.26 + 8.86e-3 * SAL + 1e-3 * TEM))
    KW = math.exp(
        +1.489802e2 - 1.384726e4 / ATEM - 2.36521e1 * math.log(ATEM)
        + ((-7.92447e1 + 3.29872e3 / ATEM + 1.20408e1 * math.log(ATEM))
           * math.sqrt(SAL) - 1.9813e-2 * SAL)
    )
    BT = 4.106e-4 * SAL / 35.0
    FH = 1.29e-2 - 2.4e-3 * ATEM + 4.61e-4 * SAL**2 - 1.48e-6 * ATEM * SAL**2
    return BT, K0, K1, K2, KB, KW, FH

def co2_nibun(BT, K0, K1, K2, KB, KW, AT, CT):
    ATX = CV2 * AT
    CTX = CV2 * CT
    HMIN, HMAX, DELTA = 1.0e-14, 1.0, 1.0e-15
    h_low, h_high = HMIN, HMAX
    hx = 0.5 * (h_low + h_high)
    for _ in range(100000):
        h = 0.5 * (h_low + h_high)
        denom = h * h + K1 * h + K1 * K2
        at_calc = (
            (2.0 * K1 * K2 * CTX) / denom
            + (h * K1 * CTX) / denom
            + (KB * BT) / (h + KB)
            + KW / h
            - h
        )
        if abs(ATX - at_calc) <= DELTA:
            hx = h
            break
        if at_calc < ATX:
            h_high = h
        else:
            h_low = h
    denom2 = hx * hx + K1 * hx + K1 * K2
    PCO2 = (hx * hx * CTX) / denom2 / K0
    CO2 = (hx * hx * CTX) / denom2 / CV2
    HCO3 = K1 * CO2 / hx
    CO32 = K2 * HCO3 / hx
    return CO2, HCO3, CO32, PCO2

def carbonate_box(temp, sal, alk, dic):
    BT, K0, K1, K2, KB, KW, FH = chemeq_const(temp, sal)
    CO2, HCO3, CO32, PCO2 = co2_nibun(BT, K0, K1, K2, KB, KW, alk, dic)
    return {"K0": K0, "CO2": CO2, "HCO3": HCO3, "CO32": CO32, "PCO2": PCO2}

In [ ]:
rows = []
for b in ["H", "L", "N", "D"]:
    c = carbonate_box(TEMP[b], SALT[b], x[f"ALK{b}"], x[f"DIC{b}"])
    rows.append({
        "Box": b,
        "Temperature": TEMP[b],
        "pCO2 [ppmv]": to_ppmv(c["PCO2"]),
        "O2 saturation [umol/kg]": to_umolkg(o2sat(TEMP[b], SALT[b])),
    })

pd.DataFrame(rows)

### Mini exercise 3 / ミニ演習 3

同じ DIC と ALK なのに、L box の pCO2 が H や N と違う理由を説明してください。  
Explain why pCO2 in the L box differs from H and N even though DIC and ALK are initially the same.

**ヒント / Hint**

炭酸系は温度に依存します。  
Carbonate chemistry depends on temperature.

## 7. 輸送の考え方 / Transport structure

4-box では、循環を次のように考えます。  
In the 4-box model, we think of circulation as follows.

```text
D → H → L → N → D
```

これは、深層水が南大洋 H で湧昇し、低緯度 L を通り、北大西洋 N で沈み込むという単純化です。  
This is a simplification in which deep water upwells in the Southern Ocean H, passes through low latitudes L, and sinks in the North Atlantic N.

さらに、H と D の間には深層ベンチレーション・混合 `FHD` を入れます。  
We also include deep ventilation/mixing `FHD` between H and D.

ここでは教育用に、輸送を次の 2 種類に分けます。  
For teaching, we separate transport into two types.

| Parameter | 意味 / Meaning |
|---|---|
| T | D → H → L → N → D の循環 |
| FHD | H と D の直接交換 |

In [ ]:
# Transport and biological parameters

DT = 8.64e4       # one day [s]
Tcir = 2.0e7      # large-scale conveyor-like circulation [m3/s]
FHD = 6.0e7       # H-D exchange [m3/s]

RCP = 106.0
RNP = 16.0
RRC = 0.20
RO2P = 172.0

# Export production parameters
CEPH = 0.75       # H export parameter [mol C m-2 yr-1]
CEPN = 0.02       # N export is weak in this teaching version
LF = 0.5
R = 1.0 / CV4
HSC = 2.0e-8 * CV1
DEL = 100.0

# Gas exchange
PVH = 3.0
PVL = 3.0
PVN = 3.0
FAH = PVH * AOCNH / CV5
FAL = PVL * AOCNL / CV5
FAN = PVN * AOCNN / CV5

EPH = (CEPH / RCP) * AOCNH / CV4
EPN = (CEPN / RCP) * AOCNN / CV4

def compute_epl(x, LF=0.5):
    return R * DEL * LF * x["PO4L"] * (x["PO4L"] / (HSC + x["PO4L"])) * VOCNL

EPL = compute_epl(x, LF=LF)

pd.DataFrame({
    "Flux": ["Tcir", "FHD", "EPH", "EPL", "EPN", "FAH", "FAL", "FAN"],
    "Value": [Tcir, FHD, EPH, EPL, EPN, FAH, FAL, FAN],
})

## 8. 4-box の新しい科学 / New science in the 4-box model

3-box では、高緯度 H が「沈み込み」と「湧昇」の両方を表していました。  
In the 3-box model, the high-latitude H box represented both sinking and upwelling.

4-box では、それを分けます。  
In the 4-box model, we separate them.

```text
N box: sinking branch / 沈み込み
H box: upwelling branch / 湧昇
```

この違いにより、次の問いを調べられます。  
This allows us to investigate:

> 南大洋の深層ベンチレーションを弱めると、大気 CO2 は下がるのか？  
> Does atmospheric CO2 decrease if Southern Ocean deep ventilation is weakened?

これは Toggweiler (1999) の中心的なアイデアにつながります。  
This connects to the central idea of Toggweiler (1999).

## 9. PO4 を 1 日だけ進める / Advance PO4 by one day

まず PO4 だけで、4-box の式の構造を見ます。  
First, we look at the structure of the 4-box equations using only PO4.

ここでは循環を次のように単純化します。  
Here we simplify the circulation as:

```text
D → H → L → N → D
```

各ボックスの PO4 は、  
For each box, PO4 changes by:

```text
transport in - transport out - biological export + remineralization
```

です。  
This is the same conservation-law idea as before.

In [ ]:
EPL = compute_epl(x, LF=LF)

PO4HX = x["PO4H"] + (
    Tcir * (x["PO4D"] - x["PO4H"])
    + FHD * (x["PO4D"] - x["PO4H"])
    - EPH
) * (DT / VOCNH)

PO4LX = x["PO4L"] + (
    Tcir * (x["PO4H"] - x["PO4L"])
    - EPL
) * (DT / VOCNL)

PO4NX = x["PO4N"] + (
    Tcir * (x["PO4L"] - x["PO4N"])
    - EPN
) * (DT / VOCNN)

PO4DX = x["PO4D"] + (
    Tcir * (x["PO4N"] - x["PO4D"])
    + FHD * (x["PO4H"] - x["PO4D"])
    + EPH + EPL + EPN
) * (DT / VOCND)

pd.DataFrame({
    "Box": ["H", "L", "N", "D"],
    "Old PO4 [umol/kg]": [to_umolkg(x[f"PO4{b}"]) for b in ["H", "L", "N", "D"]],
    "New PO4 [umol/kg]": [to_umolkg(PO4HX), to_umolkg(PO4LX), to_umolkg(PO4NX), to_umolkg(PO4DX)],
})

### Mini exercise 4 / ミニ演習 4

N box の PO4 は増えていますか、減っていますか。  
Does PO4 in the N box increase or decrease?

なぜそうなるかを、次の語を使って説明してください。  
Explain why using the following words:

```text
low-latitude water
biological export
sinking
```

```text
低緯度水
生物輸出
沈み込み
```

## 10. 1 ステップ関数を作る / Define a one-step function

ここから、PO4, DIC, ALK, O2, 大気 CO2 をまとめて進めます。  
Now we advance PO4, DIC, ALK, O2, and atmospheric CO2 together.

関数 `one_step_four_box()` は、現在の値 `x` を受け取り、1 日後の値 `y` を返します。  
The function `one_step_four_box()` receives current values `x` and returns next-day values `y`.

`x` や `y` は特別な概念ではありません。  
`x` and `y` are not special concepts.

単に、たくさんの変数をまとめる袋です。  
They are simply containers for many variables.

In [ ]:
VOLUME = {"H": VOCNH, "L": VOCNL, "N": VOCNN, "D": VOCND}
AREA = {"H": AOCNH, "L": AOCNL, "N": AOCNN}
FA = {"H": FAH, "L": FAL, "N": FAN}

def one_step_four_box(x, Tcir=2.0e7, FHD=6.0e7, LF=0.5, CEPH=0.75, CEPN=0.02, air_sea=True):
    y = dict(x)

    EPH = (CEPH / RCP) * AOCNH / CV4
    EPN = (CEPN / RCP) * AOCNN / CV4
    EPL = compute_epl(x, LF=LF)

    alk_factor = 2.0 * RRC * RCP - RNP
    dic_factor = (1.0 + RRC) * RCP

    # Gas exchange terms for DIC
    gas = {"H": 0.0, "L": 0.0, "N": 0.0}
    carb = {}
    if air_sea:
        for b in ["H", "L", "N"]:
            carb[b] = carbonate_box(TEMP[b], SALT[b], x[f"ALK{b}"], x[f"DIC{b}"])
            gas[b] = FA[b] * CV1 * carb[b]["K0"] * (x["PCO2A"] - carb[b]["PCO2"])

    exports = {"H": EPH, "L": EPL, "N": EPN}

    # Helper to update one tracer
    def update_tracer(prefix, bio_factor, gas_terms=None):
        if gas_terms is None:
            gas_terms = {"H": 0.0, "L": 0.0, "N": 0.0}

        # H: receives from D by T and FHD, loses by export
        y[f"{prefix}H"] = x[f"{prefix}H"] + (
            Tcir * (x[f"{prefix}D"] - x[f"{prefix}H"])
            + FHD * (x[f"{prefix}D"] - x[f"{prefix}H"])
            - bio_factor * exports["H"]
            + gas_terms.get("H", 0.0)
        ) * (DT / VOCNH)

        # L: receives from H, loses by export
        y[f"{prefix}L"] = x[f"{prefix}L"] + (
            Tcir * (x[f"{prefix}H"] - x[f"{prefix}L"])
            - bio_factor * exports["L"]
            + gas_terms.get("L", 0.0)
        ) * (DT / VOCNL)

        # N: receives from L, loses by export
        y[f"{prefix}N"] = x[f"{prefix}N"] + (
            Tcir * (x[f"{prefix}L"] - x[f"{prefix}N"])
            - bio_factor * exports["N"]
            + gas_terms.get("N", 0.0)
        ) * (DT / VOCNN)

        # D: receives from N by T, exchanges with H by FHD, gains remineralization
        y[f"{prefix}D"] = x[f"{prefix}D"] + (
            Tcir * (x[f"{prefix}N"] - x[f"{prefix}D"])
            + FHD * (x[f"{prefix}H"] - x[f"{prefix}D"])
            + bio_factor * (exports["H"] + exports["L"] + exports["N"])
        ) * (DT / VOCND)

    update_tracer("PO4", 1.0)
    update_tracer("ALK", alk_factor)
    update_tracer("DIC", dic_factor, gas_terms=gas)

    # O2: use opposite sign for remineralization and include simple gas restoration in surface boxes
    O2SAT = {b: o2sat(TEMP[b], SALT[b]) for b in ["H", "L", "N", "D"]}

    y["DO2H"] = x["DO2H"] + (
        Tcir * (O2SAT["D"] - x["DO2H"])
        + FHD * (O2SAT["H"] - x["DO2H"])
        + RO2P * EPH
        + FAH * (O2SAT["H"] - x["DO2H"])
    ) * (DT / VOCNH)

    y["DO2L"] = x["DO2L"] + (
        Tcir * (x["DO2H"] - x["DO2L"])
        + RO2P * EPL
        + FAL * (O2SAT["L"] - x["DO2L"])
    ) * (DT / VOCNL)

    y["DO2N"] = x["DO2N"] + (
        Tcir * (x["DO2L"] - x["DO2N"])
        + RO2P * EPN
        + FAN * (O2SAT["N"] - x["DO2N"])
    ) * (DT / VOCNN)

    y["DO2D"] = x["DO2D"] + (
        Tcir * (x["DO2N"] - x["DO2D"])
        + FHD * (x["DO2H"] - x["DO2D"])
        - RO2P * (EPH + EPL + EPN)
    ) * (DT / VOCND)

    # Atmosphere update after ocean update
    if air_sea:
        flux_to_atm = 0.0
        for b in ["H", "L", "N"]:
            cnew = carbonate_box(TEMP[b], SALT[b], y[f"ALK{b}"], y[f"DIC{b}"])
            flux_to_atm += FA[b] * CV1 * cnew["K0"] * (cnew["PCO2"] - x["PCO2A"])
        y["PCO2A"] = x["PCO2A"] + flux_to_atm * (DT / VATM)
    else:
        y["PCO2A"] = x["PCO2A"]

    diagnostics = {"EPH": EPH, "EPL": EPL, "EPN": EPN}
    return y, diagnostics

## 11. 3000 年まわす / Run the model for 3000 years

これまでと同じように、1 ステップ関数を for 文で何度も呼び出します。  
As before, we repeatedly call the one-step function using a for-loop.

1 年ごとに値を保存して、あとで図にします。  
We save values once per year and plot them later.

In [ ]:
def diagnose_four_box(x, diag=None):
    rows = {}
    for b in ["H", "L", "N", "D"]:
        c = carbonate_box(TEMP[b], SALT[b], x[f"ALK{b}"], x[f"DIC{b}"])
        rows[f"PO4{b}"] = to_umolkg(x[f"PO4{b}"])
        rows[f"DIC{b}"] = to_umolkg(x[f"DIC{b}"])
        rows[f"ALK{b}"] = to_umolkg(x[f"ALK{b}"])
        rows[f"DO2{b}"] = to_umolkg(x[f"DO2{b}"])
        rows[f"PCO2{b}"] = to_ppmv(c["PCO2"])
    rows["PCO2A"] = to_ppmv(x["PCO2A"])
    if diag is not None:
        rows["EPH_PgCyr"] = CV4 * diag["EPH"] * RCP * 12.0e-15
        rows["EPL_PgCyr"] = CV4 * diag["EPL"] * RCP * 12.0e-15
        rows["EPN_PgCyr"] = CV4 * diag["EPN"] * RCP * 12.0e-15
    return rows

def run_four_box(years=3000, Tcir=2.0e7, FHD=6.0e7, LF=0.5, CEPH=0.75, CEPN=0.02, air_sea=True):
    x = initial_four_box()
    history = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            _, diag = one_step_four_box(x, Tcir=Tcir, FHD=FHD, LF=LF, CEPH=CEPH, CEPN=CEPN, air_sea=air_sea)
            row = {"year": day / 365}
            row.update(diagnose_four_box(x, diag=diag))
            history.append(row)

        x, diag = one_step_four_box(x, Tcir=Tcir, FHD=FHD, LF=LF, CEPH=CEPH, CEPN=CEPN, air_sea=air_sea)

    return x, pd.DataFrame(history)

final, hist = run_four_box(years=3000)
hist.tail()

## 12. まず標準実験を見る / Standard experiment

標準実験で、4 つの箱がどう分かれるかを見ます。  
First, we examine how the four boxes separate in the standard experiment.

見るべき点は次です。  
Look at:

- N box の PO4 が低くなるか。  
  Does PO4 in N become low?

- D box の DIC が高くなるか。  
  Does DIC in D become high?

- H, L, N の pCO2 が違うか。  
  Are pCO2 values different among H, L, and N?

In [ ]:
def plot_boxes(hist, var, ylabel):
    plt.figure()
    for b in ["H", "L", "N", "D"]:
        plt.plot(hist["year"], hist[f"{var}{b}"], label=b)
    plt.xlabel("Time [yr]")
    plt.ylabel(ylabel)
    plt.title(var)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_boxes(hist, "PO4", "PO4 [umol/kg]")
plot_boxes(hist, "DIC", "DIC [umol/kg]")
plot_boxes(hist, "DO2", "O2 [umol/kg]")

In [ ]:
plt.figure()
plt.plot(hist["year"], hist["PCO2A"], label="Atmosphere")
for b in ["H", "L", "N", "D"]:
    plt.plot(hist["year"], hist[f"PCO2{b}"], label=b)

plt.xlabel("Time [yr]")
plt.ylabel("pCO2 [ppmv]")
plt.title("pCO2 in atmosphere and boxes")
plt.legend()
plt.grid(True)
plt.show()

## 13. 最終状態を表で見る / Final state table

最後の値を表にして、箱ごとの差を見ます。  
We summarize the final values in a table and compare boxes.

この表を見ると、N box が低栄養塩・低 pCO2 になりやすいことが分かります。  
This table helps show that the N box tends to become nutrient-poor and low in pCO2.

In [ ]:
def final_table_four_box(x):
    rows = []
    for q, prefix, unit in [
        ("PO4", "PO4", "umol/kg"),
        ("DIC", "DIC", "umol/kg"),
        ("ALK", "ALK", "ueq/kg"),
        ("O2", "DO2", "umol/kg"),
    ]:
        for b in ["H", "L", "N", "D"]:
            rows.append([q, b, to_umolkg(x[f"{prefix}{b}"]), unit])

    for b in ["H", "L", "N", "D"]:
        c = carbonate_box(TEMP[b], SALT[b], x[f"ALK{b}"], x[f"DIC{b}"])
        rows.append(["pCO2", b, to_ppmv(c["PCO2"]), "ppmv"])

    rows.append(["pCO2", "Atmosphere", to_ppmv(x["PCO2A"]), "ppmv"])

    df = pd.DataFrame(rows, columns=["Quantity", "Box", "Value", "Unit"])
    df["Value"] = df["Value"].map(lambda v: f"{v:.3f}")
    return df

final_table_four_box(final)

## 14. Challenge A: 深層ベンチレーション FHD を変える / Change deep ventilation FHD

`FHD` は H と D の交換です。  
`FHD` is the exchange between H and D.

Toggweiler 型モデルでは、深層のベンチレーションを弱めると、大気 CO2 が下がるという考え方が重要です。  
In Toggweiler-type models, a key idea is that weaker deep ventilation can lower atmospheric CO2.

**予想 / Prediction**

`FHD` を小さくすると、大気 pCO2 は上がるでしょうか、下がるでしょうか。  
If `FHD` is reduced, does atmospheric pCO2 increase or decrease?

まず予想してから実行してください。  
Predict first, then run the cell.

In [ ]:
FHD_list = [1.5e7, 3.0e7, 6.0e7, 9.0e7, 1.2e8]
summary = []

plt.figure()

for fhd in FHD_list:
    _, h = run_four_box(years=3000, FHD=fhd)
    plt.plot(h["year"], h["PCO2A"], label=f"FHD={fhd:.1e}")
    summary.append({
        "FHD": fhd,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final O2D": h["DO2D"].iloc[-1],
        "Final DICD": h["DICD"].iloc[-1],
        "Final PO4H": h["PO4H"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Sensitivity to H-D ventilation FHD")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

## 15. Challenge B: 北大西洋沈み込み T を変える / Change conveyor circulation T

`Tcir` は D → H → L → N → D の大きな循環です。  
`Tcir` represents the large-scale circulation D → H → L → N → D.

この値を小さくすると、北大西洋での沈み込みが弱くなるような実験になります。  
Reducing this value is like weakening North Atlantic sinking.

**予想 / Prediction**

`Tcir` を小さくすると、N box の PO4 と大気 pCO2 はどう変わるでしょうか。  
If `Tcir` is reduced, what happens to PO4 in N and atmospheric pCO2?

In [ ]:
T_list = [0.5e7, 1.0e7, 2.0e7, 4.0e7]
summary = []

plt.figure()

for t in T_list:
    _, h = run_four_box(years=3000, Tcir=t)
    plt.plot(h["year"], h["PO4N"], label=f"T={t:.1e}")
    summary.append({
        "Tcir": t,
        "Final PO4N": h["PO4N"].iloc[-1],
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final O2D": h["DO2D"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("N-box PO4 [umol/kg]")
plt.title("Sensitivity of N-box phosphate to Tcir")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

## 16. Challenge C: 南大洋生物ポンプ CEPH を変える / Change Southern Ocean export CEPH

`CEPH` は H box、つまり南大洋側の輸出生産を表します。  
`CEPH` controls export production in the H box, the Southern Ocean side.

Toggweiler (1999) では、南大洋の生物生産と深層ベンチレーションが、大気 CO2 の議論で重要です。  
In Toggweiler (1999), Southern Ocean biological production and deep ventilation are important for atmospheric CO2.

**予想 / Prediction**

`CEPH` を大きくすると、H box の PO4 と大気 pCO2 はどう変わるでしょうか。  
If `CEPH` is increased, what happens to H-box PO4 and atmospheric pCO2?

In [ ]:
CEPH_list = [0.1, 0.3, 0.75, 1.5, 3.0]
summary = []

plt.figure()

for ceph in CEPH_list:
    _, h = run_four_box(years=3000, CEPH=ceph)
    plt.plot(h["year"], h["PCO2A"], label=f"CEPH={ceph}")
    summary.append({
        "CEPH": ceph,
        "Final PO4H": h["PO4H"].iloc[-1],
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final EPH": h["EPH_PgCyr"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Sensitivity to Southern Ocean export")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

## 17. 自分で実験する / Your own experiment

ここでは、自分でパラメタを変えて実験します。  
Here, you change parameters and run your own experiment.

おすすめは次です。  
Recommended experiments:

```python
my_FHD = 1.5e7   # weak Southern Ocean ventilation
my_Tcir = 1.0e7  # weak conveyor circulation
my_CEPH = 3.0    # strong Southern Ocean export
my_LF = 0.8      # strong low-latitude export
```

実行する前に、大気 pCO2 が上がるか下がるかを予想してください。  
Before running, predict whether atmospheric pCO2 will increase or decrease.

In [ ]:
my_FHD = 6.0e7
my_Tcir = 2.0e7
my_CEPH = 0.75
my_LF = 0.5
my_air_sea = True

final_my, hist_my = run_four_box(
    years=3000,
    FHD=my_FHD,
    Tcir=my_Tcir,
    CEPH=my_CEPH,
    LF=my_LF,
    air_sea=my_air_sea,
)

plt.figure()
plt.plot(hist_my["year"], hist_my["PCO2A"], label="Atmosphere")
plt.plot(hist_my["year"], hist_my["PCO2H"], label="H")
plt.plot(hist_my["year"], hist_my["PCO2N"], label="N")
plt.xlabel("Time [yr]")
plt.ylabel("pCO2 [ppmv]")
plt.title("Your own 4-box experiment")
plt.legend()
plt.grid(True)
plt.show()

final_table_four_box(final_my)

## 18. 3-box から 4-box で何が増えたか / What is new from 3-box to 4-box?

3-box から 4-box への拡張で増えたのは、単なる箱の数ではありません。  
The extension from 3-box to 4-box is not just an increase in the number of boxes.

増えたのは、**問いの種類**です。  
What increases is the **type of questions we can ask**.

3-box で聞けた問い:

```text
高緯度ベンチレーションを変えると大気 CO2 はどう変わるか？
```

Questions possible in the 3-box model:

```text
What happens to atmospheric CO2 if high-latitude ventilation changes?
```

4-box で聞けるようになった問い:

```text
南大洋の湧昇と北大西洋の沈み込みを分けるとどうなるか？
```

Questions possible in the 4-box model:

```text
What happens if Southern Ocean upwelling and North Atlantic sinking are separated?
```

これがモデル拡張の本質です。  
This is the essence of model expansion.

## 19. 課題 / Exercises

### 課題 1 / Exercise 1

3-box モデルで高緯度を 1 つの箱にした場合、何が表現できないか。  
What cannot be represented when high latitudes are treated as one box in the 3-box model?

### 課題 2 / Exercise 2

4-box モデルで N box を追加する科学的な意味を説明せよ。  
Explain the scientific meaning of adding the N box in the 4-box model.

### 課題 3 / Exercise 3

`FHD` を小さくしたとき、大気 pCO2、深層 DIC、深層 O2 はどう変化するか。  
When `FHD` is reduced, what happens to atmospheric pCO2, deep DIC, and deep O2?

### 課題 4 / Exercise 4

`Tcir` を小さくしたとき、N box の PO4 はどう変化するか。  
When `Tcir` is reduced, what happens to PO4 in the N box?

### 課題 5 / Exercise 5

4-box でもまだ表現できない海洋過程を 3 つ挙げよ。  
List three ocean processes that still cannot be represented by the 4-box model.

---

## 想定解答 / Expected answers

### 課題 1

3-box では、高緯度が 1 つの箱なので、北大西洋の沈み込みと南大洋の湧昇を区別できない。  
In the 3-box model, high latitudes are represented by one box, so North Atlantic sinking and Southern Ocean upwelling cannot be distinguished.

### 課題 2

N box は北大西洋を表し、低緯度から来た水が沈み込む場所を表す。これにより、深層水形成と南大洋湧昇を別々に扱える。  
The N box represents the North Atlantic, where water from low latitudes sinks. This allows deep-water formation and Southern Ocean upwelling to be treated separately.

### 課題 3

`FHD` を小さくすると、H と D の交換が弱まり、深層がより孤立する。多くの場合、深層 DIC は増え、深層 O2 は減り、大気 pCO2 は下がる方向に動く。  
Reducing `FHD` weakens exchange between H and D and makes the deep ocean more isolated. In many cases, deep DIC increases, deep O2 decreases, and atmospheric pCO2 tends to decrease.

### 課題 4

`Tcir` を小さくすると、低緯度から N box へ運ばれる水が減るため、N box の栄養塩供給が変化する。N box は低栄養塩のままになりやすいが、応答は輸出生産やガス交換にも依存する。  
Reducing `Tcir` changes the supply of water from low latitudes to the N box. The N box tends to remain nutrient-poor, but the response also depends on export production and gas exchange.

### 課題 5

4-box でも、中層水、太平洋と大西洋の違い、インド洋、海氷、季節変化、空間的に連続した南大洋の構造などは表現できない。  
Even the 4-box model cannot represent intermediate waters, Atlantic-Pacific differences, the Indian Ocean, sea ice, seasonality, or the continuous spatial structure of the Southern Ocean.

## 20. 次へ / Next step

4-box モデルでは、北大西洋 N と南大洋 H を分けることができました。  
In the 4-box model, we separated the North Atlantic N and Southern Ocean H.

しかし、まだ深層は 1 つの箱です。  
However, the deep ocean is still one box.

次の 6-box モデルでは、深層の内部構造を分けます。  
In the next 6-box model, we divide the deep ocean into internal layers.

```text
4-box:
  surface regions + one deep box

6-box:
  surface regions + intermediate/deep structure
```

これにより、より現実的な海洋循環とトレーサー分布に近づきます。  
This brings us closer to realistic ocean circulation and tracer distributions.